# Q2 — Reconstructed Genetic Algorithm with Further Iterations and Convergence

This notebook follows the **same calculation style as the handwritten solution**.

### Given in the handwritten solution

- Objective function: \(f(x)=-x^2+2x\)
- Initial population:
  `11010, 00111, 10110, 00101`
- Random numbers:
  `0.4, 0.15, 0.7, 0.9`
- Search interval: \(0\le x\le2\)
- 5-bit chromosomes
- The handwritten solution calculates:
  - \(x\)-value from the binary chromosome
  - \(f(x)\)
  - probability
  - cumulative probability
  - roulette-wheel selection
  - mating pool
- The handwritten page gives the next population as:
  `10110, 01010, 00111, 10101`

The notebook **keeps that handwritten Population 2 exactly as shown**, then continues the same selection/reproduction process for additional generations and checks convergence.

> The exact operation that produced the handwritten Population 2 is not fully visible in the photograph, so it is not invented in the notebook. Population 2 is therefore treated as the supplied result from the handwritten solution.


In [ ]:
import pandas as pd
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

# ============================================================
# USER INPUT
# ============================================================

function_expression = input(
    "Enter function f(x) [press Enter for -x^2 + 2*x]: "
).strip()

if function_expression == "":
    function_expression = "-x^2 + 2*x"

x_symbol = sp.symbols("x")
expression = sp.sympify(function_expression.replace("^", "**"))
fitness_function = sp.lambdify(x_symbol, expression, "math")

print(f"Using f(x) = {function_expression}")


In [ ]:
# ============================================================
# FIXED QUESTION DATA
# ============================================================

INITIAL_POPULATION = [
    "11010",
    "00111",
    "10110",
    "00101"
]

RANDOM_NUMBERS = [0.4, 0.15, 0.7, 0.9]

X_MIN = 0
X_MAX = 2
BITS = 5

# Handwritten solution's Population 2
HANDWRITTEN_POPULATION_2 = [
    "10110",
    "01010",
    "00111",
    "10101"
]

print("Initial population :", INITIAL_POPULATION)
print("Random numbers     :", RANDOM_NUMBERS)
print("Handwritten Pop. 2 :", HANDWRITTEN_POPULATION_2)


## 1. Binary chromosome → decimal → x-value

For a 5-bit chromosome:

\[
D=\text{decimal value of chromosome}
\]

and

\[
x=x_{min}+\frac{x_{max}-x_{min}}{2^5-1}D
\]

Since \(x_{min}=0\) and \(x_{max}=2\),

\[
x=\frac{2D}{31}.
\]

For example:

\[
11010_2=26
\]

so

\[
x=\frac{2(26)}{31}=1.6774.
\]


In [ ]:
def decimal_value(chromosome):
    return int(chromosome, 2)

def x_value(chromosome):
    d = decimal_value(chromosome)
    return X_MIN + ((X_MAX - X_MIN) / (2**BITS - 1)) * d

def make_table(population):
    df = pd.DataFrame({"Chromosome": population})
    df["Decimal"] = df["Chromosome"].apply(decimal_value)
    df["x-value"] = df["Chromosome"].apply(x_value)
    df["f(x)"] = df["x-value"].apply(fitness_function)

    # The handwritten solution uses f(x)/sum(f(x)) as probability.
    total_fitness = df["f(x)"].sum()
    df["Probability"] = df["f(x)"] / total_fitness
    df["Cumulative Probability"] = df["Probability"].cumsum()

    return df

population_1_table = make_table(INITIAL_POPULATION)

population_1_table.round(4)


In [ ]:
print(f"Σ f(x) = {population_1_table['f(x)'].sum():.6f}")
print(f"Average = {population_1_table['f(x)'].mean():.6f}")
print(f"Maximum = {population_1_table['f(x)'].max():.6f}")


## 2. Roulette-wheel selection

The random numbers are used exactly as in the handwritten solution.

For each random number \(r\), select the first chromosome whose cumulative probability is at least \(r\).


In [ ]:
def roulette_selection(population_table, random_numbers):
    selected = []

    for r in random_numbers:
        selected_chromosome = None

        for _, row in population_table.iterrows():
            if r <= row["Cumulative Probability"]:
                selected_chromosome = row["Chromosome"]
                break

        selected.append(selected_chromosome)

    return selected

mating_pool_1 = roulette_selection(
    population_1_table,
    RANDOM_NUMBERS
)

selection_table_1 = pd.DataFrame({
    "Random Number": RANDOM_NUMBERS,
    "Selected Chromosome": mating_pool_1
})

selection_table_1


## 3. Mating pool

The resulting mating pool is displayed separately, matching the style of the handwritten calculation.


In [ ]:
print("MATING POOL")
print("-----------")

for i, chromosome in enumerate(mating_pool_1, start=1):
    print(f"{i}. {chromosome}")


## 4. Population 2 — handwritten result

The photograph shows the following new population:

| S.No | New population |
|---:|:---:|
| 1 | `10110` |
| 2 | `01010` |
| 3 | `00111` |
| 4 | `10101` |

We preserve this result rather than assuming an unseen crossover/mutation operation.


In [ ]:
population_2 = HANDWRITTEN_POPULATION_2.copy()

population_2_table = make_table(population_2)

population_2_table.round(4)


In [ ]:
print(f"Σ f(x), Population 2 = {population_2_table['f(x)'].sum():.6f}")
print(f"Average, Population 2 = {population_2_table['f(x)'].mean():.6f}")
print(f"Maximum, Population 2 = {population_2_table['f(x)'].max():.6f}")


# 5. Continue to further generations

From Population 2 onward, the notebook repeats the **visible selection procedure**:

**Population → f(x) → probability → cumulative probability → given random numbers → mating pool**

The crossover point stated in the question is the **5th digit**. A cut after the 5th bit leaves a 5-bit chromosome unchanged, so for the continuation we explicitly show the mating pool as the next population.

This is useful because it lets us determine whether the **fitness has converged** even if the order of chromosomes changes.


In [ ]:
MAX_GENERATIONS = 20

# A generation is considered fitness-converged when the best fitness
# has remained unchanged (within numerical tolerance) for 3 consecutive generations.
STABLE_BEST_GENERATIONS = 3

history = []

current_population = population_2.copy()
best_fitness_history = []

for generation in range(2, MAX_GENERATIONS + 1):

    current_table = make_table(current_population)

    best_fitness = current_table["f(x)"].max()
    average_fitness = current_table["f(x)"].mean()

    history.append({
        "Generation": generation,
        "Population": current_population.copy(),
        "Best f(x)": best_fitness,
        "Average f(x)": average_fitness,
        "Best chromosome": current_table.loc[
            current_table["f(x)"].idxmax(), "Chromosome"
        ]
    })

    best_fitness_history.append(best_fitness)

    # Selection using the same four random numbers
    mating_pool = roulette_selection(
        current_table,
        RANDOM_NUMBERS
    )

    # Crossover point = 5th digit.
    # Cutting after the last bit does not alter the chromosome.
    next_population = mating_pool.copy()

    current_population = next_population

    if len(best_fitness_history) >= STABLE_BEST_GENERATIONS:
        recent = best_fitness_history[-STABLE_BEST_GENERATIONS:]

        if np.allclose(
            recent,
            recent[0],
            rtol=0,
            atol=1e-10
        ):
            break

history_df = pd.DataFrame(history)

history_df[[
    "Generation",
    "Best chromosome",
    "Best f(x)",
    "Average f(x)"
]].round(6)


## 6. Generation-by-generation result

This table shows the same type of convergence information as the previous notebook.


In [ ]:
display_table = history_df[[
    "Generation",
    "Best chromosome",
    "Best f(x)",
    "Average f(x)"
]].copy()

display_table.round(6)


## 7. Convergence condition

For this notebook, the convergence condition is:

> **The best fitness remains unchanged for 3 consecutive generations.**

We also check whether the complete chromosome population becomes identical between consecutive generations.

These are kept separate because a GA can have **fitness convergence without population convergence**.


In [ ]:
best_values = history_df["Best f(x)"].to_numpy()

fitness_converged = False

if len(best_values) >= STABLE_BEST_GENERATIONS:
    recent = best_values[-STABLE_BEST_GENERATIONS:]
    fitness_converged = np.allclose(
        recent,
        recent[0],
        rtol=0,
        atol=1e-10
    )

population_converged = False

for i in range(1, len(history)):
    if history[i]["Population"] == history[i-1]["Population"]:
        population_converged = True
        break

print("CONVERGENCE CHECK")
print("=================")
print("Fitness convergence :", fitness_converged)
print("Population convergence :", population_converged)

if fitness_converged:
    print(
        f"Best fitness remained approximately "
        f"{best_values[-1]:.6f} for the last "
        f"{STABLE_BEST_GENERATIONS} generations."
    )
else:
    print("Fitness convergence condition was not reached.")


## 8. Plot convergence

The graph shows:

- Best fitness by generation
- Average fitness by generation

This makes the convergence behaviour easy to see.


In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    history_df["Generation"],
    history_df["Best f(x)"],
    marker="o",
    label="Best f(x)"
)

plt.plot(
    history_df["Generation"],
    history_df["Average f(x)"],
    marker="s",
    label="Average f(x)"
)

plt.xlabel("Generation")
plt.ylabel("Fitness")
plt.title(f"GA Convergence — f(x) = {function_expression}")
plt.xticks(history_df["Generation"])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 9. Final generation


In [ ]:
final_generation = history[-1]["Generation"]
final_population = history[-1]["Population"]
final_table = make_table(final_population)

print(f"Final generation: {final_generation}")
print("Final population:")
for i, chromosome in enumerate(final_population, start=1):
    print(f"{i}. {chromosome}")

print()
print(f"Best chromosome = {final_table.loc[final_table['f(x)'].idxmax(), 'Chromosome']}")
print(f"Best x-value    = {final_table['x-value'].max():.6f}")
print(f"Best f(x)       = {final_table['f(x)'].max():.6f}")

final_table.round(6)
